<a href="https://colab.research.google.com/github/NojoodAlghamdi-AI/Chatbot-breakdown-detector-/blob/main/CHAT_INTL_D_fullDBDC3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CHAT-INTL-D: Prompt-Based Conversation Breakdown Detection
# Version 3 — Full dialogue context Ct at each turn
# Two-stage: Zero-shot screening + Instruction-based refinement
# Outputs: Breakdown flag, Breakdown category, Confidence score
# Checkpoint/resume every 100 rows
# ============================================================

# !pip install --upgrade openai pandas openpyxl

import pandas as pd
import os
import time
from openai import OpenAI
from io import BytesIO

# ============================================================
# CONFIGURATION
# ============================================================

client = OpenAI(api_key="YOUR_API_KEY_HERE")  # Replace with your key

TEMPERATURE      = 0
API_DELAY        = 1
MODEL            = "gpt-4o"
CHECKPOINT_EVERY = 100
CHECKPOINT_FILE  = "chatintl_checkpoint.csv"

# ============================================================
# STAGE 1: ZERO-SHOT PROMPT — with full dialogue context
# ============================================================

def build_zero_shot_prompt(context: str, user: str, bot: str) -> str:
    return f"""You are a dialogue quality evaluator. Your task is to determine whether a chatbot response constitutes a conversation breakdown, given the full preceding conversation.

A breakdown occurs when the chatbot's response is:
- Semantically irrelevant to the user's query or conversation topic
- Misaligned with the user's intent
- Repetitive or circular — looping prior content without advancing the conversation
- Internally incoherent or contradictory with earlier turns
- A generic fallback where a substantive response was expected

Important: Evaluate the bot response in the context of the full conversation history below. A response that seems odd in isolation may be acceptable given the conversation context — and vice versa. Only classify as BREAKDOWN if the response genuinely fails given the full context.

--- Conversation history ---
{context if context.strip() else '[No prior context]'}

--- Turn being evaluated ---
User: {user if user.strip() else '[no message]'}
Bot: {bot if bot.strip() else '[no response]'}

Respond in this exact format:
Classification: [BREAKDOWN or NO BREAKDOWN]
Confidence: [a number between 0.0 and 1.0]"""


# ============================================================
# STAGE 2: INSTRUCTION-BASED REFINEMENT — with full context
# ============================================================

def build_instruction_based_prompt(context: str, user: str, bot: str) -> str:
    return f"""You are an expert dialogue quality analyst. A preliminary evaluation has flagged the following chatbot response as a potential breakdown. Using the full conversation history, conduct a step-by-step assessment against each criterion. For each criterion, state whether it is satisfied or violated. Then provide a final classification.

Criteria:
1. Semantic relevance: Does the response address the topic raised by the user, given the conversation context?
2. Intent alignment: Does the response address what the user was actually seeking in this conversation?
3. Logical coherence: Is the response internally consistent and free of contradictions with prior turns?
4. Substantive contribution: Does the response advance the conversation with meaningful content?

--- Conversation history ---
{context if context.strip() else '[No prior context]'}

--- Turn being evaluated ---
User: {user if user.strip() else '[no message]'}
Bot: {bot if bot.strip() else '[no response]'}

If the final classification is BREAKDOWN, identify which category best describes the failure:
- Semantic Irrelevance
- Misunderstanding of User Intent
- Repetitive or Circular Response
- Incoherent or Contradictory Response
- Failure to Respond

Respond in this exact format:
Criterion 1 - Semantic relevance: [satisfied / violated]
Criterion 2 - Intent alignment: [satisfied / violated]
Criterion 3 - Logical coherence: [satisfied / violated]
Criterion 4 - Substantive contribution: [satisfied / violated]
Final classification: [BREAKDOWN or NO BREAKDOWN]
Confidence: [a number between 0.0 and 1.0]
Breakdown category: [category name or N/A if NO BREAKDOWN]"""


# ============================================================
# OUTPUT PARSERS
# ============================================================

def parse_zero_shot_output(text: str) -> dict:
    result = {"stage1_classification": "Error", "stage1_confidence": None}
    for line in text.splitlines():
        line = line.strip()
        if line.lower().startswith("classification:"):
            val = line.split(":", 1)[1].strip().upper()
            result["stage1_classification"] = "NO BREAKDOWN" if "NO BREAKDOWN" in val else (
                "BREAKDOWN" if "BREAKDOWN" in val else "Error")
        elif line.lower().startswith("confidence:"):
            try:
                result["stage1_confidence"] = float(line.split(":", 1)[1].strip())
            except ValueError:
                pass
    return result


def parse_instruction_based_output(text: str) -> dict:
    result = {
        "final_classification": "Error", "confidence": None,
        "breakdown_category": "N/A",
        "criterion_1": None, "criterion_2": None,
        "criterion_3": None, "criterion_4": None
    }
    for line in text.splitlines():
        line = line.strip()
        if line.lower().startswith("criterion 1"):
            result["criterion_1"] = "violated" if "violated" in line.lower() else "satisfied"
        elif line.lower().startswith("criterion 2"):
            result["criterion_2"] = "violated" if "violated" in line.lower() else "satisfied"
        elif line.lower().startswith("criterion 3"):
            result["criterion_3"] = "violated" if "violated" in line.lower() else "satisfied"
        elif line.lower().startswith("criterion 4"):
            result["criterion_4"] = "violated" if "violated" in line.lower() else "satisfied"
        elif line.lower().startswith("final classification:"):
            val = line.split(":", 1)[1].strip().upper()
            result["final_classification"] = "NO BREAKDOWN" if "NO BREAKDOWN" in val else (
                "BREAKDOWN" if "BREAKDOWN" in val else "Error")
        elif line.lower().startswith("confidence:"):
            try:
                result["confidence"] = float(line.split(":", 1)[1].strip())
            except ValueError:
                pass
        elif line.lower().startswith("breakdown category:"):
            cat = line.split(":", 1)[1].strip()
            result["breakdown_category"] = "N/A" if cat.lower() == "n/a" else cat
    return result


# ============================================================
# LLM INVOCATION
# ============================================================

def call_llm(system_msg: str, user_msg: str) -> str:
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user",   "content": user_msg}
            ],
            temperature=TEMPERATURE
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"\n    ⚠️  LLM error: {e}")
        return "Error"


# ============================================================
# TWO-STAGE DETECTION — with dialogue context Ct
# ============================================================

def detect_breakdown(context: str, user: str, bot: str) -> dict:
    """
    Two-stage CHAT-INTL-D detection on a single turn (Ct, ut, rt).

    Inputs:
      context : full dialogue history up to turn t (Ct)
      user    : user message at turn t (ut)
      bot     : chatbot response at turn t (rt)

    Stage 1 — Zero-shot screening with full context.
    Stage 2 — Instruction-based refinement (flagged cases only).
    Two-criterion confirmation rule: requires ≥2 criteria violated.
    """
    output = {
        "Final_Classification":  None,
        "Confidence":            None,
        "Breakdown_Category":    "N/A",
        "Stage1_Classification": None,
        "Stage1_Confidence":     None,
        "Criterion_1":           "N/A",
        "Criterion_2":           "N/A",
        "Criterion_3":           "N/A",
        "Criterion_4":           "N/A",
        "Stage2_Applied":        False
    }

    # Absent response → immediate BREAKDOWN
    if not bot.strip():
        output.update({
            "Final_Classification":  "BREAKDOWN",
            "Confidence":            1.0,
            "Breakdown_Category":    "Failure to Respond",
            "Stage1_Classification": "BREAKDOWN",
            "Stage1_Confidence":     1.0
        })
        return output

    # ── Stage 1: Zero-shot screening ──────────────────────────────────
    zs_raw = call_llm(
        system_msg="You are a helpful assistant detecting chatbot breakdowns.",
        user_msg=build_zero_shot_prompt(context, user, bot)
    )
    time.sleep(API_DELAY)

    if zs_raw == "Error":
        output["Final_Classification"]  = "Error"
        output["Stage1_Classification"] = "Error"
        return output

    zs = parse_zero_shot_output(zs_raw)
    output["Stage1_Classification"] = zs["stage1_classification"]
    output["Stage1_Confidence"]     = zs["stage1_confidence"]

    if zs["stage1_classification"] == "NO BREAKDOWN":
        output["Final_Classification"] = "NO BREAKDOWN"
        output["Confidence"]           = zs["stage1_confidence"]
        return output

    # ── Stage 2: Instruction-based refinement ─────────────────────────
    output["Stage2_Applied"] = True

    ib_raw = call_llm(
        system_msg="You are an expert dialogue quality analyst.",
        user_msg=build_instruction_based_prompt(context, user, bot)
    )
    time.sleep(API_DELAY)

    if ib_raw == "Error":
        output["Final_Classification"] = zs["stage1_classification"]
        output["Confidence"]           = zs["stage1_confidence"]
        return output

    ib = parse_instruction_based_output(ib_raw)
    output["Criterion_1"]        = ib["criterion_1"]
    output["Criterion_2"]        = ib["criterion_2"]
    output["Criterion_3"]        = ib["criterion_3"]
    output["Criterion_4"]        = ib["criterion_4"]
    output["Breakdown_Category"] = ib["breakdown_category"]

    # Two-criterion confirmation rule
    violations = sum(
        1 for c in [ib["criterion_1"], ib["criterion_2"],
                    ib["criterion_3"], ib["criterion_4"]]
        if c == "violated"
    )

    if violations >= 2 and ib["final_classification"] == "BREAKDOWN":
        output["Final_Classification"] = "BREAKDOWN"
        output["Confidence"]           = ib["confidence"]
    else:
        output["Final_Classification"] = "NO BREAKDOWN"
        output["Confidence"]           = ib["confidence"]
        output["Breakdown_Category"]   = "N/A"

    return output


# ============================================================
# CHECKPOINT HELPERS
# ============================================================

def load_checkpoint(f):
    if os.path.exists(f):
        df = pd.read_csv(f)
        print(f"🔄 Checkpoint found — resuming from row {len(df)}")
        return df
    return pd.DataFrame()

def save_checkpoint(results, f):
    pd.DataFrame(results).to_csv(f, index=False)

def delete_checkpoint(f):
    if os.path.exists(f):
        os.remove(f)
        print("🗑️  Checkpoint deleted (run complete).")


# ============================================================
# MAIN PIPELINE
# ============================================================

def run_detection_pipeline(input_path: str, output_path: str):
    """
    Load the DBDC3 eval Excel (with Context_Ct column), apply
    two-stage detection to every row, save results.

    Required columns: Context_Ct, User_Message, Bot_Response
    Optional column:  Label (ground truth)
    """
    xls = pd.ExcelFile(input_path)

    for sheet in xls.sheet_names:
        df         = xls.parse(sheet)
        total_rows = len(df)
        print(f"\n📋 Sheet: '{sheet}' — {total_rows} rows")

        required = {"Context_Ct", "User_Message", "Bot_Response"}
        if not required.issubset(df.columns):
            print(f"  ❌ Missing columns: {required - set(df.columns)}")
            continue

        # Load checkpoint
        ckpt_df      = load_checkpoint(CHECKPOINT_FILE)
        already_done = len(ckpt_df)
        results      = ckpt_df.to_dict("records") if already_done > 0 else []

        if already_done >= total_rows:
            print(f"  ✅ All rows already done (checkpoint).")
        else:
            if already_done > 0:
                print(f"  ⏩ Skipping first {already_done} rows.")

            for idx in range(already_done, total_rows):
                row     = df.iloc[idx]
                context = str(row.get("Context_Ct",    "")).strip()
                user    = str(row.get("User_Message",  "")).strip()
                bot     = str(row.get("Bot_Response",  "")).strip()

                print(f"  Row {idx+1}/{total_rows} "
                      f"({round((idx+1)/total_rows*100,1)}%) ...", end="\r")

                detection = detect_breakdown(context, user, bot)
                results.append(detection)

                if (idx + 1) % CHECKPOINT_EVERY == 0:
                    save_checkpoint(results, CHECKPOINT_FILE)
                    print(f"\n  💾 Checkpoint saved — {idx+1}/{total_rows} done.")

            print(f"\n  ✅ All rows processed.")

        # Merge and save
        df_out = pd.concat(
            [df.reset_index(drop=True),
             pd.DataFrame(results).reset_index(drop=True)],
            axis=1
        )

        buf = BytesIO()
        with pd.ExcelWriter(buf, engine="openpyxl") as writer:
            df_out.to_excel(writer, sheet_name=sheet, index=False)
        buf.seek(0)
        with open(output_path, "wb") as f:
            f.write(buf.read())

        print(f"  📁 Saved to: '{output_path}'")
        delete_checkpoint(CHECKPOINT_FILE)


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":

    # ── Google Colab ───────────────────────────────────────────────────
    from google.colab import files
    uploaded    = files.upload()
    input_file  = list(uploaded.keys())[0]
    output_file = "CHAT_INTL_D_v3_results.xlsx"

    # ── Local use: comment above 3 lines, uncomment below ─────────────
    # input_file  = "DBDC3_eval_full_context.xlsx"
    # output_file = "CHAT_INTL_D_v3_results.xlsx"

    run_detection_pipeline(input_file, output_file)

    # ── Google Colab download ──────────────────────────────────────────
    files.download(output_file)

Saving DBDC3_eval_full_context.xlsx to DBDC3_eval_full_context.xlsx

📋 Sheet: 'Sheet1' — 2000 rows

  💾 Checkpoint saved — 100/2000 done.

  💾 Checkpoint saved — 200/2000 done.

  💾 Checkpoint saved — 300/2000 done.

  💾 Checkpoint saved — 400/2000 done.

  💾 Checkpoint saved — 500/2000 done.

  💾 Checkpoint saved — 600/2000 done.

  💾 Checkpoint saved — 700/2000 done.

  💾 Checkpoint saved — 800/2000 done.

  💾 Checkpoint saved — 900/2000 done.

  💾 Checkpoint saved — 1000/2000 done.

  💾 Checkpoint saved — 1100/2000 done.

  💾 Checkpoint saved — 1200/2000 done.

  💾 Checkpoint saved — 1300/2000 done.

  💾 Checkpoint saved — 1400/2000 done.

  💾 Checkpoint saved — 1500/2000 done.

  💾 Checkpoint saved — 1600/2000 done.

  💾 Checkpoint saved — 1700/2000 done.

  💾 Checkpoint saved — 1800/2000 done.

  💾 Checkpoint saved — 1900/2000 done.

  💾 Checkpoint saved — 2000/2000 done.

  ✅ All rows processed.
  📁 Saved to: 'CHAT_INTL_D_v3_results.xlsx'
🗑️  Checkpoint deleted (run complete).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>